In [1]:
pip -q install langgraph langchain langchain-core langchain-groq groq

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Digite sua GROQ_API_KEY: ")
print("✅ GROQ_API_KEY configurada!")

✅ GROQ_API_KEY configurada!


In [2]:
import os
import re
from getpass import getpass

from typing_extensions import TypedDict
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command



In [3]:
def make_llm(temperature: float = 0.7) -> ChatGroq:
    return ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=temperature,
    )

llm = make_llm()
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C199A608C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C199CF8980>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [29]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt
from langchain_core.messages import SystemMessage, HumanMessage


class FeedbackState(TypedDict, total=False):
    tema: str
    nivel: str
    roteiro: str
    pedido: str
    versao_final: str


def node_planejar(state: FeedbackState) -> FeedbackState:
    tema = state.get("tema", "tema")
    nivel = state.get("nivel", "iniciante")

    resp = llm.invoke([
        SystemMessage(content="Escreva em PT-BR, de acordo com o nível solicitado."),
        HumanMessage(content=(
            f"Crie um roteiro em 5 tópicos para uma aula sobre {tema}. "
            f"Nível: {nivel}. Inclua 1 exemplo prático."
        )),
    ])

    return {"roteiro": resp.content}


def node_pedir_feedback(state: FeedbackState) -> FeedbackState:
    novo_nivel = interrupt({
        "pergunta": (
            "Qual nível deseja? iniciante/intermediário/avançado. "
            "Deixe em branco para manter."
        ),
        "nivel_atual": state.get("nivel", "iniciante"),
    })

    nivel = state.get("nivel", "iniciante")

    if isinstance(novo_nivel, str) and novo_nivel.strip():
        nivel = novo_nivel.strip().casefold()

    tema = state.get("tema", "tema")

    while True:
        novo_tema = interrupt({
            "pergunta": "Qual tema deseja? Deixe em branco para manter.",
            "tema_atual": tema,
        })

        if isinstance(novo_tema, str) and novo_tema.strip():
            tema = novo_tema.strip()

        confirmacao = interrupt({
            "pergunta": f"Tem certeza do tema '{tema}'? Responda yes ou no.",
            "tema": tema,
        })

        while str(confirmacao).strip().casefold() not in {"yes", "no"}:
            confirmacao = interrupt({
                "pergunta": "Resposta inválida. Responda yes ou no.",
                "tema": tema,
            })

        if str(confirmacao).strip().casefold() == "yes":
            return {"nivel": nivel, "tema": tema}


def node_finalizar(state: FeedbackState) -> FeedbackState:
    resp = llm.invoke([
        SystemMessage(content=(
            "Escreva em PT-BR, com estrutura para apostila. "
            "Respeite rigorosamente o nível solicitado."
        )),
        HumanMessage(content=(
            "Transforme o roteiro abaixo em uma versão final formatada.\n"
            "Adapte o conteúdo ao nível informado e atenda ao pedido "
            "adicional do humano.\n\n"
            f"TEMA: {state.get('tema')}\n"
            f"NÍVEL: {state.get('nivel')}\n\n"
            f"ROTEIRO:\n{state.get('roteiro', '')}\n\n"
            f"PEDIDO ADICIONAL: {state.get('pedido') or 'Nenhum.'}\n\n"
            "Inclua uma seção final: 'Erros comuns' com 3 itens."
        )),
    ])

    return {"versao_final": resp.content}


fb_builder = StateGraph(FeedbackState)

fb_builder.add_node("planejar", node_planejar)
fb_builder.add_node("feedback", node_pedir_feedback)
fb_builder.add_node("replanejar", node_planejar)
fb_builder.add_node("finalizar", node_finalizar)

fb_builder.add_edge(START, "planejar")
fb_builder.add_edge("planejar", "feedback")
fb_builder.add_edge("feedback", "replanejar")
fb_builder.add_edge("replanejar", "finalizar")
fb_builder.add_edge("finalizar", END)

fb_checkpointer = MemorySaver()
fb_graph = fb_builder.compile(checkpointer=fb_checkpointer)

In [30]:
fb_config = {
    "configurable": {
        "thread_id": "fb-tres-nos-02"
    }
}

# 1. Inicia com nível iniciante
parcial = fb_graph.invoke(
    {
        "tema": "Computação Quântica",
        "nivel": "iniciante"
    },
    config=fb_config,
)

print(parcial.get("__interrupt__"))


# 2. Altera o nível para avançado
parcial = fb_graph.invoke(
    Command(resume="avançado"),
    config=fb_config,
)

print("Nível atual:", parcial.get("nivel"))
print(parcial.get("__interrupt__"))


# 3. Altera o tema
parcial = fb_graph.invoke(
    Command(
        resume="Computação Quântica em processamentos comparados com GPU"
    ),
    config=fb_config,
)

print(parcial.get("__interrupt__"))


# 4. Confirma o tema e gera a apostila
final = fb_graph.invoke(
    Command(resume="yes"),
    config=fb_config,
)

print("Nível:", final["nivel"])
print("Tema:", final["tema"])
print(final["versao_final"])

[Interrupt(value={'pergunta': 'Qual nível deseja? iniciante/intermediário/avançado. Deixe em branco para manter.', 'nivel_atual': 'iniciante'}, id='1425b8735075e8814c2aab2a8559e37e', response_schema=None)]
Nível atual: iniciante
[Interrupt(value={'pergunta': 'Qual tema deseja? Deixe em branco para manter.', 'tema_atual': 'Computação Quântica'}, id='1425b8735075e8814c2aab2a8559e37e', response_schema=None)]
[Interrupt(value={'pergunta': "Tem certeza do tema 'Computação Quântica em processamentos comparados com GPU'? Responda yes ou no.", 'tema': 'Computação Quântica em processamentos comparados com GPU'}, id='1425b8735075e8814c2aab2a8559e37e', response_schema=None)]
Nível: avançado
Tema: Computação Quântica em processamentos comparados com GPU
# **Apostila – Computação Quântica vs. Processamento em GPUs**  
**Nível:** Avançado | **Carga horária:** 2 h (inclui demonstração prática)  

---  

## Sumário  

1. [Fundamentos da Computação Quântica e da Arquitetura de GPUs](#1-fundamentos)  
2